In [ ]:
import pandas as pd
import sys
sys.path.append('../../')


# load data
Crick_H1N1 = pd.read_excel('../../../data/raw/data4model(Crick-H1N1).xlsx')
Crick_H3N2 = pd.read_excel('../../../data/raw/data4model(Crick-H3N2).xlsx')
origin_df = pd.concat([Crick_H1N1, Crick_H3N2]).reset_index(drop=True)

## DNA MEGA

In [6]:
## select required columns
AA_data_filt1 = origin_df[['serumName','virusName','serumHA', 'serumNA', 'virusHA', 'virusNA', 
                           'serumPassCat','virusPassCat', 'serumType','HI_Dist']].copy()
## remove duplicated row and mean HI_Dist
AA_data_filt2 = AA_data_filt1.groupby(['serumHA', 'serumNA', 'virusHA', 'virusNA', 'serumPassCat', 'virusPassCat']) \
        .agg({'serumName': 'first', 'virusName': 'first', 'serumType': 'first', 'HI_Dist': 'mean'}) \
        .reset_index()[['serumName', 'virusName', 'serumHA', 'serumNA', 'virusHA', 'virusNA',
                        'serumPassCat', 'virusPassCat', 'serumType', 'HI_Dist']]
## remove PassCat = 'BOTH'
AA_data_filt3 = AA_data_filt2[(AA_data_filt2['serumPassCat'] != 'BOTH') &
                              (AA_data_filt2['virusPassCat'] != 'BOTH')].reset_index(drop=True)
## replace PassCat to special token
AA_data_filt4 = AA_data_filt3.replace({'serumPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'},
                                       'virusPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'}})

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader

class AADataset(Dataset):
    def __init__(self, DataFrame):
        self.sequence = (DataFrame['serumHA'] + '<eos>' + DataFrame['serumNA'] + '<eos>' + DataFrame['virusHA'] + '<eos>' + DataFrame['virusNA'] + \
                         '<eos>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat']).tolist()
        self.labels = torch.tensor(DataFrame['HI_Dist'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.sequence[idx], self.labels[idx]

In [4]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(AA_data_filt4, test_size=0.1, random_state=42)
train_df, valid_df = train_test_split(train_df, test_size=1/9, random_state=42)

train_dataset = AADataset(train_df)
valid_dataset = AADataset(valid_df)
test_dataset = AADataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=80, shuffle=False)

In [5]:
# train_df.to_csv('../../../data/processed/1.1/AA_train_df.csv', index=True)
# valid_df.to_csv('../../../data/processed/1.1/AA_valid_df.csv', index=True)
# test_df.to_csv('../../../data/processed/1.1/AA_test_df.csv', index=True)

In [ ]:
from bio_tokenizer import BioTokenizer
from transformers import MegaConfig, MegaForSequenceClassification
from MEGA_utilities import count_parameters
from torch.optim import AdamW
from transformers import get_scheduler
import torch

# get tokenizer and model
tokenizer = BioTokenizer(vocab_file='./vocab_AA.txt')

# update the num_vocab and num_label
config = MegaConfig()
config.num_labels=1
config.vocab_size=28
config.max_positions=4000
config.num_attention_heads=4
config.num_hidden_layers=5
device = torch.device("cuda:1")
model = MegaForSequenceClassification(config)
model.to(device)
print("Number of parameters: %e"%count_parameters(model))

# optimizer
optimizer = AdamW(model.parameters(), lr=5e-4)

# scheduler
num_epochs = 160
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(name="linear", optimizer=optimizer,
                             num_warmup_steps=len(train_loader), num_training_steps=num_training_steps)

Number of parameters: 1.134667e+06


In [7]:
from tqdm import tqdm
from utilities import print_exams
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from utilities import EarlyStopping
from datetime import datetime

progress_bar = tqdm(range(num_training_steps))
early_stopping = EarlyStopping(patience=10, delta=0.005, save_dir='../../../trained_model/1.2_HANA_model_new/')

# Training loop
for epoch in range(num_epochs):
    model.train()
    loss_ls = []
    for batch_seq, batch_label in train_loader:
        batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
        batch_input = batch_input.to(device)
        batch_label = batch_label.to(device)

        outputs = model(**batch_input, labels=batch_label)

        loss = outputs.loss
        loss_ls.append(loss.item())

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
    train_loss = sum(loss_ls) / len(loss_ls)
    print('train loss :', train_loss)

    prediction_ls = []
    reference_ls = []
    logits_ls = []
    loss_ls_valid = []
    with torch.no_grad():
        model.eval()
        for batch_seq, batch_label in valid_loader:
            batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
            batch_input = batch_input.to(device)
            batch_label = batch_label.to(device)

            outputs = model(**batch_input, labels=batch_label)
            logits = outputs.logits
            loss = outputs.loss

            logits_ls.append(logits)
            loss_ls_valid.append(loss.item())
            prediction_ls += logits.tolist()
            prediction_ls_final = []
            for sublist in prediction_ls:
                for element in sublist:
                    prediction_ls_final.append(element)
            reference_ls += batch_label.tolist()

    print_exams(prediction_ls_final, reference_ls)
    valid_MAE = mean_absolute_error(reference_ls, prediction_ls_final)
    valid_mse = mean_squared_error(reference_ls, prediction_ls_final)
    valid_pearson = pearsonr(reference_ls, prediction_ls_final).statistic
    valid_spearman = spearmanr(reference_ls, prediction_ls_final).statistic
    
    early_stopping(valid_mse, model)
    if early_stopping.early_stop:
        print("Early stopping")
        break

    ## 将epoch信息写入log.txt
    with open('../../../trained_model/1.2_HANA_model_new/log.txt', 'a') as f:
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"[{current_time}] Epoch {epoch + 1}/{num_epochs}, train loss: {train_loss:.4f}, valid MAE: {valid_MAE:.4f}, valid MSE: {valid_mse:.4f}, valid Pearson: {valid_pearson:.4f}, valid Spearman: {valid_spearman:.4f}\n")

  1%|          | 5761/921760 [17:38<45:49:50,  5.55it/s]

train loss : 3.003582780903008
MAE:  1.3333299372874436
MSE:  3.06556192014959
pearson correlation:  PearsonRResult(statistic=0.49005657376315825, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.5183472299511013, pvalue=0.0)
R2_score:  0.1912335257684269
Validation MSE decrease (inf --> 3.065562).  Saving model ...


  1%|▏         | 11522/921760 [36:45<44:28:24,  5.69it/s] 

train loss : 2.952435884624073
MAE:  1.3204242219350981
MSE:  2.8865444773054643
pearson correlation:  PearsonRResult(statistic=0.4892190281325944, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.5227426714174054, pvalue=0.0)
R2_score:  0.23846248732465958
Validation MSE decrease (3.065562 --> 2.886544).  Saving model ...


  2%|▏         | 17283/921760 [55:53<44:08:02,  5.69it/s]  

...

train loss : 0.5586521607283564


 86%|████████▋ | 795019/921760 [43:55:19<944:26:30, 26.83s/it]

MAE:  0.6822099843951104
MSE:  0.788790910596284
pearson correlation:  PearsonRResult(statistic=0.890239054120912, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.8631767561901229, pvalue=0.0)
R2_score:  0.7918986273036238
EarlyStopping counter: 9 out of 10


 87%|████████▋ | 800779/921760 [44:12:53<5:52:39,  5.72it/s]  

train loss : 0.5604838976962112
MAE:  0.6806866698874449
MSE:  0.787876691945102
pearson correlation:  PearsonRResult(statistic=0.8914512638986254, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.864480122881534, pvalue=0.0)
R2_score:  0.7921398194290652
EarlyStopping counter: 10 out of 10
Early stopping


In [11]:
from bio_tokenizer import BioTokenizer
from utilities import print_exams
import torch

prediction_ls = []
reference_ls = []
logits_ls = []
loss_ls_valid = []

tokenizer = BioTokenizer(vocab_file='./vocab_AA.txt')
device = torch.device('cuda:1')
model = torch.load('../../../trained_model/1.2_HANA_model_new/2025-07-19_10-53-19.pth', weights_only=False, map_location=device)
model.eval()
for batch_seq, batch_label in valid_loader:
    batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
    batch_input = batch_input.to(device)
    batch_label = batch_label.to(device)
    with torch.no_grad():
        outputs = model(**batch_input, labels=batch_label)
    logits = outputs.logits
    loss = outputs.loss

    logits_ls.append(logits)
    loss_ls_valid.append(loss.item())
    prediction_ls += logits.tolist()
    prediction_ls_final = []
    for sublist in prediction_ls:
        for element in sublist:
            prediction_ls_final.append(element)
    reference_ls += batch_label.tolist()

print_exams(prediction_ls_final, reference_ls)

MAE:  0.6779233298936067
MSE:  0.7794392907296332
pearson correlation:  PearsonRResult(statistic=0.8926168447273077, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.8651883949093491, pvalue=0.0)
R2_score:  0.794365801944003


In [8]:
from bio_tokenizer import BioTokenizer
from utilities import print_exams
import torch

prediction_ls = []
reference_ls = []
logits_ls = []
loss_ls_valid = []

tokenizer = BioTokenizer(vocab_file='./vocab_AA.txt')
device = torch.device('cuda:1')
model = torch.load('../../../trained_model/1.2_HANA_model_new/2025-07-19_10-53-19.pth', weights_only=False, map_location=device)
model.eval()
for batch_seq, batch_label in test_loader:
    batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
    batch_input = batch_input.to(device)
    batch_label = batch_label.to(device)
    with torch.no_grad():
        outputs = model(**batch_input, labels=batch_label)
    logits = outputs.logits
    loss = outputs.loss

    logits_ls.append(logits)
    loss_ls_valid.append(loss.item())
    prediction_ls += logits.tolist()
    prediction_ls_final = []
    for sublist in prediction_ls:
        for element in sublist:
            prediction_ls_final.append(element)
    reference_ls += batch_label.tolist()

print_exams(prediction_ls_final, reference_ls)

MAE: 0.61797
MSE: 0.70557
pearson correlation: 0.90011
spearman correlation: 0.86934
R2_score: 0.80900


(0.6179685946287751,
 0.7055740593939708,
 PearsonRResult(statistic=0.9001129952198043, pvalue=0.0),
 SignificanceResult(statistic=0.8693438555608725, pvalue=0.0),
 0.8090043627200303)

In [7]:
from bio_tokenizer import BioTokenizer
from utilities import print_exams
import torch

prediction_ls = []
reference_ls = []
logits_ls = []
loss_ls_valid = []

tokenizer = BioTokenizer(vocab_file='./vocab_AA.txt')
device = torch.device('cuda:1')
model = torch.load('../../../trained_model/1.2_HANA_model_new/2025-07-19_10-53-19.pth', weights_only=False, map_location=device)
model.eval()
for batch_seq, batch_label in valid_loader:
    batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
    batch_input = batch_input.to(device)
    batch_label = batch_label.to(device)
    with torch.no_grad():
        outputs = model(**batch_input, labels=batch_label)
    logits = outputs.logits
    loss = outputs.loss

    logits_ls.append(logits)
    loss_ls_valid.append(loss.item())
    prediction_ls += logits.tolist()
    prediction_ls_final = []
    for sublist in prediction_ls:
        for element in sublist:
            prediction_ls_final.append(element)
    reference_ls += batch_label.tolist()

print_exams(prediction_ls_final, reference_ls)

MAE: 0.61789
MSE: 0.68847
pearson correlation: 0.90104
spearman correlation: 0.86845
R2_score: 0.81117


(0.617890420807107,
 0.6884695665380479,
 PearsonRResult(statistic=0.9010370210691954, pvalue=0.0),
 SignificanceResult(statistic=0.8684494635969174, pvalue=0.0),
 0.81116525754609)